# qbraid.runtime: Equal1 Bell-1

[Equal1](https://www.equal1.com) builds quantum processors from **electron spins in silicon**.

**Bell-1** is the Equal1 emulator of that hardware. It is a state vector simulator with **6 qubits**. Its noise models come from characterization data that Equal1 measured on real devices.

This notebook does four things:

1. It connects to Bell-1 and lists the noise models.
2. It runs one circuit without noise and with noise.
3. It compares three noise models on a GHZ circuit.
4. It submits several circuits as one batch job, and it sets the other runtime options.

You need an Equal1 API key. Create one at [account.equal1.com/account/api-keys](https://account.equal1.com/account/api-keys). Bell-1 is currently **free to run**. `device.metadata()` shows the current price of a job.

In [ ]:
%%capture

# "%%capture" hides the install output. Remove it if you need to debug this step.
%pip install --upgrade 'qbraid[qiskit,visualization]'

## Connect to the device

Bell-1 jobs go through the Equal1 API, so the session needs two things: your Equal1 API key and the Equal1 base URL.

The next cell reads the key from the `EQUAL1_API_KEY` environment variable. If the variable is not set, the cell asks you for the key. Do not write the key into the notebook.

In [ ]:
import os
from getpass import getpass

from qbraid.runtime import QbraidProvider
from qbraid_core import QbraidSessionV1
from qbraid_core.services.runtime import QuantumRuntimeClient

api_key = os.getenv("EQUAL1_API_KEY") or getpass("Equal1 API key: ")

session = QbraidSessionV1(api_key=api_key, base_url="https://api.equal1.com/api/v1")
provider = QbraidProvider(client=QuantumRuntimeClient(session=session))
device = provider.get_device("qbraid:equal1:sim:bell-1")

device.metadata()

The metadata shows the device status, the number of qubits, and the price.

The metadata also lists the noise models. Each noise model has a name, and you select a noise model by that name.

In [ ]:
noise_models = list(device.profile.noise_models)

noise_models

This notebook uses three of the noise models:

| Noise model | What it simulates |
|-------------|-------------------|
| `ideal` | No noise. Use it to check that your algorithm is correct. |
| `bell1-6-no-spam` | Gate noise only. It leaves out the state preparation and measurement (SPAM) errors. |
| `bell1-6` | The full noise of the Bell-1 hardware: gate noise and SPAM errors. |

## Build a circuit

A Bell pair is the smallest circuit that shows entanglement. A perfect device returns only `00` and `11`.

In [ ]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()

circuit.draw()

## Run the circuit without noise

Device options for Bell-1 go in a **`runtime_options` dict**. The `noise_model` key selects the noise model.

```python
device.run(circuit, shots=1000, noise_model="ideal")                      # wrong
device.run(circuit, shots=1000, runtime_options={"noise_model": "ideal"}) # correct
```

`shots` is the exception. You pass `shots` directly to `run()`.

In [ ]:
ideal_job = device.run(circuit, shots=1000, runtime_options={"noise_model": "ideal"})
ideal_counts = ideal_job.result().data.get_counts()

ideal_counts

The result has two outcomes, `00` and `11`. Each outcome has about half of the shots.

## Run the circuit with the Bell-1 noise model

The circuit and the number of shots stay the same. Only the noise model changes.

`run()` removes the `noise_model` key from the dict that you pass. Build a new dict for every call. Do not reuse one dict for two jobs.

In [ ]:
noisy_job = device.run(circuit, shots=1000, runtime_options={"noise_model": "bell1-6"})
noisy_counts = noisy_job.result().data.get_counts()

noisy_counts

Compare these counts with the counts from the `ideal` run. The outcomes `01` and `10` are errors, because a perfect Bell pair never produces them.

A Bell pair has one two-qubit gate, so the noise has little to act on. A deeper circuit shows the noise more clearly.

## Compare the noise models on a GHZ circuit

A GHZ circuit on `n` qubits needs `n - 1` two-qubit gates. A perfect device returns only the all-zeros outcome and the all-ones outcome.

The next cell runs a 5-qubit GHZ circuit under each of the three noise models. It then measures the share of shots that give a valid GHZ outcome.

In [ ]:
def ghz(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc


def ghz_fraction(counts: dict[str, int], n: int) -> float:
    valid = counts.get("0" * n, 0) + counts.get("1" * n, 0)
    return valid / sum(counts.values())


NUM_QUBITS = 5
SHOTS = 1000
models = ["ideal", "bell1-6-no-spam", "bell1-6"]

model_counts = {}
for model in models:
    job = device.run(ghz(NUM_QUBITS), shots=SHOTS, runtime_options={"noise_model": model})
    model_counts[model] = job.result().data.get_counts()

for model, counts in model_counts.items():
    print(f"{model:16s} valid GHZ outcomes: {ghz_fraction(counts, NUM_QUBITS):.1%}")

Read the three numbers in order:

- `ideal` gives 100%.
- `bell1-6-no-spam` gives a lower number. The difference from `ideal` is the gate noise.
- `bell1-6` gives the lowest number. The difference from `bell1-6-no-spam` is the SPAM errors.

This comparison tells you where the errors in your circuit come from.

The histogram shows the same three results:

In [ ]:
from qbraid.visualization import plot_histogram

plot_histogram(
    [model_counts[model] for model in models],
    legend=models,
    title=f"{NUM_QUBITS}-qubit GHZ circuit on Equal1 Bell-1",
)

## Submit several circuits as one batch job

Bell-1 supports batch jobs. A batch job sends a list of circuits in one API call, and it returns one job.

Pass the list to `run()` with `as_batch=True`. One noise model applies to every circuit in the batch job.

The next cell measures how the noise grows with the size of the GHZ circuit. The largest circuit uses all 6 qubits.

In [ ]:
sizes = [2, 3, 4, 5, 6]

batch_job = device.run(
    [ghz(n) for n in sizes],
    shots=SHOTS,
    as_batch=True,
    runtime_options={"noise_model": "bell1-6"},
)
batch_result = batch_job.result()  # BatchResult

print(f"circuits in the batch job: {batch_result.num_circuits}")

for n, circuit_result in zip(sizes, batch_result.results):
    counts = circuit_result.data.get_counts()
    print(f"{n} qubits: valid GHZ outcomes {ghz_fraction(counts, n):.1%}")

`batch_result.results` holds one result for each circuit. The results keep the order of the list that you submitted.

Use a batch job for a parameter sweep. One batch job is faster than many separate jobs.

## Set the other runtime options

`runtime_options` accepts two more keys:

| Key | Values | Default | What it does |
|-----|--------|---------|--------------|
| `simulationPlatform` | `"CPU"`, `"GPU"` | `"CPU"` | It selects the processor that runs the simulation. |
| `execution_options` | dict | | It controls how Bell-1 compiles your circuit. |

`execution_options` has two settings:

- `optimization_level` sets how much the Bell-1 compiler optimizes the circuit. The value `3` is the highest level.
- `force_bypass_transpilation` skips the Bell-1 compiler when you set it to `1`. Your circuit must then already use gates that Bell-1 accepts.

The next cell sets every option in one job. Change `"CPU"` to `"GPU"` to run the simulation on a GPU.

In [ ]:
runtime_options = {
    "noise_model": "bell1-6",
    "simulationPlatform": "CPU",
    "execution_options": {
        "force_bypass_transpilation": 0,
        "optimization_level": 3,
    },
}

options_job = device.run(ghz(NUM_QUBITS), shots=SHOTS, runtime_options=runtime_options)
options_counts = options_job.result().data.get_counts()

print(f"valid GHZ outcomes: {ghz_fraction(options_counts, NUM_QUBITS):.1%}")

## Where next

- [`qbraid_runtime_batch_jobs.ipynb`](qbraid_runtime_batch_jobs.ipynb), in this folder, covers batch jobs in more detail.
- The [batch jobs documentation](https://docs.qbraid.com/v2/sdk/user-guide/providers/native/jobs#batch-jobs) describes `BatchResult`.
- The [Equal1 website](https://www.equal1.com) describes the Bell-1 hardware.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>